# 1. Historical Stock Price Scraping for IDX30 Constituents

This notebook automates the extraction of historical daily stock prices from **Yahoo Finance** (`yfinance`) for all candidate stocks listed in the Indonesia Stock Exchange (IDX) **IDX30** index across the study period (2022-12-30 to 2026-01-27).

The extracted dataset contains daily closing prices used as the foundational raw input for portfolio optimization and walk-forward backtesting.

### Import Libraries

In [1]:
import os
import pandas as pd
import yfinance as yf

### Load IDX30 Constituents List

In [2]:
list_path = "../data/raw/IDX30-List.csv"
df_idx30 = pd.read_csv(list_path)
print(f"Total unique candidate tickers in IDX30 historical periods: {df_idx30[ticker].nunique()}")
df_idx30.head()

### Define Date Parameters & Yahoo Finance Tickers

In [3]:
START_DATE = "2022-12-30"
END_DATE = "2026-01-27"

tickers = df_idx30["ticker"].unique()
tickers_yahoo = [f"{ticker}.JK" for ticker in tickers]

print(f"Observation window: {START_DATE} to {END_DATE}")
print(f"Number of target tickers: {len(tickers_yahoo)}")

### Define Download Function

In [4]:
def download_stock_data(ticker, start_date, end_date):
    """Download daily closing prices from Yahoo Finance."""
    df = yf.download(
        ticker,
        start=start_date,
        end=end_date,
        interval="1d",
        progress=False
    )
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    if df.empty:
        return None
    df = df[["Close"]].reset_index()
    df["ticker"] = ticker.replace(".JK", "")
    df = df[["Date", "ticker", "Close"]]
    return df

### Scrape Historical Prices & Save to `data/raw/`

In [5]:
OUTPUT_FILE = "../data/raw/IDX30-Scraping.csv"

# Check if dataset already exists to avoid redundant network scraping
if os.path.exists(OUTPUT_FILE):
    print(f"Dataset already available at {OUTPUT_FILE}.")
    df_all = pd.read_csv(OUTPUT_FILE)
    print(f"Total records loaded: {len(df_all):,}")
else:
    all_data = []
    for ticker in tickers_yahoo:
        print(f"Fetching {ticker}...")
        data = download_stock_data(ticker, START_DATE, END_DATE)
        if data is not None:
            all_data.append(data)
    df_all = pd.concat(all_data, ignore_index=True)
    df_all.to_csv(OUTPUT_FILE, index=False)
    print(f"Dataset successfully saved to {OUTPUT_FILE} ({len(df_all):,} records).")

df_all.head()